In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 70'S REAL POLICY + ALL 13
#            PRIOR PROBLEMS' REAL SUMMARY JSONS (FRESH, THIS KERNEL)
# =============================================================================
import json
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 70's Real Policy")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P14_ROOT = PROJECT_ROOT / "Phase5_Customer_Business_Intelligence" / "Problem14_Executive_Decision_Support_Dashboard"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB70_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_70_summary.json"
for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first."),
    (NB70_SUMMARY_PATH, "run 70_executive_dashboard_business_understanding.ipynb first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected location.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB70_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB70_SUMMARY = json.load(f)

WARP_THREAD_COUNT = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
MAX_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
RANDOM_SEED = NB70_SUMMARY["random_seed"]
PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}

POLICY_PATH = Path(NB70_SUMMARY["policy_path"])
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    EXECUTIVE_DASHBOARD_POLICY = json.load(f)

# Real, canonical registry -- read from Notebook 70's own persisted policy JSON
# (single source of truth; not re-typed). Keys arrive as strings from JSON;
# normalize back to int problem numbers for convenient indexing.
PRIOR_PROBLEMS_REGISTRY = {int(k): v for k, v in EXECUTIVE_DASHBOARD_POLICY["prior_problems_registry"].items()}
EXECUTIVE_KPI_TARGETS = EXECUTIVE_DASHBOARD_POLICY["kpi_targets"]
EXPECTED_FOUNDATIONAL_PROBLEMS = set(EXECUTIVE_DASHBOARD_POLICY["expected_foundational_problems"])
EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS = set(EXECUTIVE_DASHBOARD_POLICY["expected_reserve_optimization_problems"])
PROBLEM_ARTIFACTS_FALLBACK_DIRS = {
    int(k): Path(v) for k, v in EXECUTIVE_DASHBOARD_POLICY["problem_artifacts_fallback_dirs"].items()
}


def _resolve_summary_json_path(pnum: int, fname: str) -> Path:
    # Real path resolution: Problems 1-4 (Phases 1-2, run earliest) wrote
    # their real summary JSONs to their own per-problem artifacts/ subfolder,
    # before this platform standardized on the shared top-level folder --
    # both locations are real, sourced from Notebook 70's own policy.
    _shared = ARTIFACTS_DIR / fname
    if _shared.exists():
        return _shared
    _fallback_dir = PROBLEM_ARTIFACTS_FALLBACK_DIRS.get(pnum)
    if _fallback_dir is not None:
        _fallback_path = _fallback_dir / fname
        if _fallback_path.exists():
            return _fallback_path
    return _shared


# --- Load every real summary JSON for all 13 problems, fresh in this kernel.
LOADED_SUMMARIES = {}
_missing = []
for _pnum, _entry in PRIOR_PROBLEMS_REGISTRY.items():
    LOADED_SUMMARIES[_pnum] = {}
    for _key, _fname in _entry["summary_jsons"].items():
        _path = _resolve_summary_json_path(_pnum, _fname)
        if not _path.exists():
            _missing.append(f"Problem {_pnum}: {_fname}")
            continue
        with open(_path, "r", encoding="utf-8") as f:
            LOADED_SUMMARIES[_pnum][_key] = json.load(f)
if _missing:
    raise FileNotFoundError("Missing real summary JSON(s):\n  " + "\n  ".join(_missing))

print(f"Real summary JSONs re-loaded (fresh kernel) for all {len(PRIOR_PROBLEMS_REGISTRY)} prior problems.")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
if missing:
    raise ImportError(f"Missing required libraries: {missing}. Install with: pip install {' '.join(missing)}")

print("✅ Section 2 complete.")


# =============================================================================
# SECTION 3: EXTRACTION HELPER (SAME PATTERN FOR EVERY PROBLEM -- NO SPECIAL
#            CASING PER PROBLEM, DRIVEN ENTIRELY BY THE REGISTRY'S OWN PATHS)
# =============================================================================
_section("SECTION 3: Extraction Helper")


def _extract(problem_summaries: dict, path):
    """Walk a (source_key, *nested_keys) path through a problem's loaded summary
    dict(s). Returns None if the path is absent at any level -- never raises,
    so a problem with a genuinely different real schema degrades to a clearly
    labeled N/A rather than crashing the whole rollup."""
    if path is None:
        return None
    source_key, *nested = path
    node = problem_summaries.get(source_key)
    for key in nested:
        if not isinstance(node, dict):
            return None
        node = node.get(key)
    return node


print("✅ Section 3 complete: _extract() will drive every field pull below, purely from the registry's "
      "own declared paths -- zero per-problem special-casing in this section.")


# =============================================================================
# SECTION 4: BUILD THE REAL EXECUTIVE ROLLUP TABLE (THE BI AGGREGATION LAYER)
# =============================================================================
_section("SECTION 4: Build the Real Executive Rollup Table")

EXECUTIVE_ROWS = []
for _pnum in sorted(PRIOR_PROBLEMS_REGISTRY):
    _entry = PRIOR_PROBLEMS_REGISTRY[_pnum]
    _summaries = LOADED_SUMMARIES[_pnum]
    _quality_val = _extract(_summaries, _entry["model_quality_path"])
    _financial_val = _extract(_summaries, _entry["financial_field"])
    _status_val = _extract(_summaries, _entry["status_field"])
    EXECUTIVE_ROWS.append({
        "problem_number": _pnum, "problem_name": _entry["problem_name"], "phase": _entry["phase"],
        "category": _entry["category"],
        "model_quality_label": _entry["model_quality_label"], "model_quality_value": _quality_val,
        "financial_label": _entry["financial_label"],
        "financial_value_usd": (round(_financial_val, 2) if isinstance(_financial_val, (int, float))
                                 else None),
        "recommended_for_production": (bool(_status_val) if isinstance(_status_val, bool) else None),
        "notes": _entry["notes"],
    })

EXECUTIVE_TABLE = pl.DataFrame(EXECUTIVE_ROWS)
print(EXECUTIVE_TABLE.select(
    ["problem_number", "problem_name", "category", "financial_value_usd", "recommended_for_production"]
))
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: KPI 1 -- AGGREGATION_COMPLETENESS
# =============================================================================
_section("SECTION 5: KPI 1 -- aggregation_completeness")

# Every real problem_number in the registry must have produced a row, and every
# value_creation problem's financial_value_usd must be a real, non-null number
# (a value_creation problem with a null benefit is a real data problem, not an
# acceptable N/A -- unlike foundational/reserve_optimization problems, where a
# null financial_value_usd is expected and correct).
_all_13_present = len(EXECUTIVE_ROWS) == 13 and set(r["problem_number"] for r in EXECUTIVE_ROWS) == set(range(1, 14))
_value_creation_rows = [r for r in EXECUTIVE_ROWS if r["category"] == "value_creation"]
_value_creation_all_have_financials = all(r["financial_value_usd"] is not None for r in _value_creation_rows)
AGGREGATION_COMPLETENESS_PASSED = bool(_all_13_present and _value_creation_all_have_financials)

print(f"All 13 problems present                              : {_all_13_present}")
print(f"Every value_creation problem has a real financial figure : {_value_creation_all_have_financials}")
print(f"aggregation_completeness: {'PASS' if AGGREGATION_COMPLETENESS_PASSED else 'FAIL'}")
if not AGGREGATION_COMPLETENESS_PASSED:
    raise RuntimeError("aggregation_completeness KPI failed -- see printed detail above.")
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: KPI 2 -- AGGREGATION_SCOPE_CORRECTNESS
# =============================================================================
_section("SECTION 6: KPI 2 -- aggregation_scope_correctness")

INCLUDED_PROBLEMS = set(
    r["problem_number"] for r in EXECUTIVE_ROWS
    if r["category"] == "value_creation" and r["recommended_for_production"] is True
)
EXCLUDED_PROBLEMS = set(range(1, 14)) - INCLUDED_PROBLEMS

# (a) Partition completeness -- included and excluded together cover all 13
#     problems exactly once, no overlap, nothing left out.
_check_a = (INCLUDED_PROBLEMS | EXCLUDED_PROBLEMS) == set(range(1, 14)) and not (INCLUDED_PROBLEMS &
                                                                                  EXCLUDED_PROBLEMS)
# (b) Every foundational/reserve_optimization problem -- a category fixed at
#     DESIGN TIME in Notebook 70, independent of any runtime recommended_for_
#     production flag -- is excluded, regardless of what that flag says.
_check_b = (EXPECTED_FOUNDATIONAL_PROBLEMS | EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS).issubset(EXCLUDED_PROBLEMS)
# (c) Every value_creation problem excluded from the total is excluded FOR A
#     REAL, DATA-DRIVEN REASON: its own real recommended_for_production flag
#     is not True (never excluded silently/arbitrarily).
_value_creation_excluded = EXCLUDED_PROBLEMS - EXPECTED_FOUNDATIONAL_PROBLEMS - EXPECTED_RESERVE_OPTIMIZATION_PROBLEMS
_check_c = all(
    next(r for r in EXECUTIVE_ROWS if r["problem_number"] == p)["recommended_for_production"] is not True
    for p in _value_creation_excluded
)
# (d) Arithmetic consistency: the headline total equals the exact sum over the
#     included set, computed independently of how INCLUDED_PROBLEMS was built.
_recompute_total = sum(
    r["financial_value_usd"] for r in EXECUTIVE_ROWS if r["problem_number"] in INCLUDED_PROBLEMS
)

AGGREGATION_SCOPE_CORRECTNESS_PASSED = bool(_check_a and _check_b and _check_c)

print(f"(a) Partition completeness (included ∪ excluded == all 13, no overlap): {_check_a}")
print(f"(b) All foundational/reserve_optimization problems excluded           : {_check_b}")
print(f"(c) Every value_creation exclusion is data-driven (real flag != True) : {_check_c}")
print(f"Included problems (value_creation, real recommended_for_production)   : {sorted(INCLUDED_PROBLEMS)}")
print(f"Excluded problems                                                     : {sorted(EXCLUDED_PROBLEMS)}")
for _p in sorted(EXCLUDED_PROBLEMS):
    _row = next(r for r in EXECUTIVE_ROWS if r["problem_number"] == _p)
    _reason = ("foundational model -- no benefit figure to sum" if _row["category"] == "foundational_model"
               else "reserve-optimization figure -- different kind of number, kept separate" if
               _row["category"] == "reserve_optimization" else
               "real recommended_for_production is not True -- not deployed, correctly excluded")
    print(f"    Problem {_p:>2} ({_row['problem_name']}): {_reason}")
print(f"aggregation_scope_correctness: {'PASS' if AGGREGATION_SCOPE_CORRECTNESS_PASSED else 'FAIL'}")
if not AGGREGATION_SCOPE_CORRECTNESS_PASSED:
    raise RuntimeError("aggregation_scope_correctness KPI failed -- see printed detail above.")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: TOTAL PLATFORM NET VALUE (THE REAL HEADLINE FIGURE)
# =============================================================================
_section("SECTION 7: Total Platform Net Value")

TOTAL_PLATFORM_NET_VALUE_USD = round(_recompute_total, 2)
RESERVE_OPTIMIZATION_VALUE_USD = round(sum(
    r["financial_value_usd"] for r in EXECUTIVE_ROWS
    if r["category"] == "reserve_optimization" and r["financial_value_usd"] is not None
), 2)
N_PROBLEMS_RECOMMENDED = len(INCLUDED_PROBLEMS)
N_PROBLEMS_FOUNDATIONAL = len(EXPECTED_FOUNDATIONAL_PROBLEMS)
N_PROBLEMS_NOT_RECOMMENDED = len(_value_creation_excluded)

print(f"TOTAL_PLATFORM_NET_VALUE_USD (per cycle, {N_PROBLEMS_RECOMMENDED} recommended value-creation "
      f"problems, kept distinct from exposure/reserve figures) : ${TOTAL_PLATFORM_NET_VALUE_USD:,.2f}")
print(f"RESERVE_OPTIMIZATION_VALUE_USD (Problem 3, kept separate)                                 : "
      f"${RESERVE_OPTIMIZATION_VALUE_USD:,.2f}")
print(f"Foundational models (no benefit figure, value realized downstream)                        : "
      f"{N_PROBLEMS_FOUNDATIONAL}")
print(f"Value-creation problems built but NOT recommended for production (excluded, shown for "
      f"transparency)                                                                              : "
      f"{N_PROBLEMS_NOT_RECOMMENDED}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: REAL PORTFOLIO-POPULATION CROSS-CHECK
# =============================================================================
_section("SECTION 8: Real Portfolio-Population Cross-Check")

# P12 and P13 both report a real "eligible_population" field independently
# (Notebook 65 and Notebook 69 respectively) -- they should agree exactly,
# since both score the same real Problem-10 worklist population. A genuinely
# new cross-problem consistency check this platform has not run before.
_p12_pop = LOADED_SUMMARIES[12]["financial"].get("eligible_population")
_p13_pop = LOADED_SUMMARIES[13]["financial"].get("eligible_population")
POPULATION_CONSISTENCY_PASSED = bool(_p12_pop is not None and _p12_pop == _p13_pop)

print(f"Problem 12 real eligible_population : {_p12_pop:,}" if _p12_pop is not None else "Problem 12: N/A")
print(f"Problem 13 real eligible_population : {_p13_pop:,}" if _p13_pop is not None else "Problem 13: N/A")
_pop_verdict = "PASS" if POPULATION_CONSISTENCY_PASSED else "FAIL -- investigate before trusting cross-problem totals"
print(f"population_consistency (informational, not hard-gating): {_pop_verdict}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: PERSIST THE REAL EXECUTIVE AGGREGATION ARTIFACTS
# =============================================================================
_section("SECTION 9: Persist the Real Executive Aggregation Artifacts")

if "executive_dashboard_modeling" in PILLAR_DIRS:
    P14_MODELING_DIR = PILLAR_DIRS["executive_dashboard_modeling"]
else:
    P14_MODELING_DIR = P14_ROOT / "modeling"
    print(f"NOTE: 'executive_dashboard_modeling' not in pillar_dirs -- using fallback: {P14_MODELING_DIR}")
P14_MODELING_DIR.mkdir(parents=True, exist_ok=True)

TABLE_PATH = P14_MODELING_DIR / "executive_aggregation_table.parquet"
EXECUTIVE_TABLE.write_parquet(TABLE_PATH)
CSV_PATH = P14_MODELING_DIR / "executive_aggregation_table.csv"
EXECUTIVE_TABLE.write_csv(CSV_PATH)

EXECUTIVE_DASHBOARD_DATA = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "rows": EXECUTIVE_ROWS,
    "total_platform_net_value_usd": TOTAL_PLATFORM_NET_VALUE_USD,
    "reserve_optimization_value_usd": RESERVE_OPTIMIZATION_VALUE_USD,
    "included_problems": sorted(INCLUDED_PROBLEMS),
    "excluded_problems": sorted(EXCLUDED_PROBLEMS),
    "n_problems_recommended": N_PROBLEMS_RECOMMENDED,
    "n_problems_foundational": N_PROBLEMS_FOUNDATIONAL,
    "n_problems_not_recommended": N_PROBLEMS_NOT_RECOMMENDED,
    "population_consistency_passed": POPULATION_CONSISTENCY_PASSED,
    "p12_eligible_population": _p12_pop, "p13_eligible_population": _p13_pop,
}
DASHBOARD_DATA_PATH = P14_MODELING_DIR / "executive_dashboard_data.json"
with open(DASHBOARD_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(EXECUTIVE_DASHBOARD_DATA, f, indent=2)

MODELING_RESULTS = {
    "table_path": str(TABLE_PATH), "csv_path": str(CSV_PATH), "dashboard_data_path": str(DASHBOARD_DATA_PATH),
    "total_platform_net_value_usd": TOTAL_PLATFORM_NET_VALUE_USD,
    "reserve_optimization_value_usd": RESERVE_OPTIMIZATION_VALUE_USD,
    "included_problems": sorted(INCLUDED_PROBLEMS), "excluded_problems": sorted(EXCLUDED_PROBLEMS),
    "kpi_results": {
        "aggregation_completeness": {"passed": AGGREGATION_COMPLETENESS_PASSED},
        "aggregation_scope_correctness": {"passed": AGGREGATION_SCOPE_CORRECTNESS_PASSED},
    },
    "population_consistency_passed": POPULATION_CONSISTENCY_PASSED,
}
MODELING_RESULTS_PATH = P14_MODELING_DIR / "executive_dashboard_modeling_results.json"
with open(MODELING_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(MODELING_RESULTS, f, indent=2)

print(f"Table written to      : {TABLE_PATH}")
print(f"Dashboard data written to : {DASHBOARD_DATA_PATH}")
print(f"Modeling results written to : {MODELING_RESULTS_PATH}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE NOTEBOOK 71 SUMMARY
# =============================================================================
_section("SECTION 10: Write Notebook 71 Summary")

notebook_71_summary = {
    "notebook": "71_executive_dashboard_modeling",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "modeling_results_path": str(MODELING_RESULTS_PATH),
    "dashboard_data_path": str(DASHBOARD_DATA_PATH),
    "table_path": str(TABLE_PATH),
    "total_platform_net_value_usd": TOTAL_PLATFORM_NET_VALUE_USD,
    "reserve_optimization_value_usd": RESERVE_OPTIMIZATION_VALUE_USD,
    "n_problems_recommended": N_PROBLEMS_RECOMMENDED,
    "n_problems_foundational": N_PROBLEMS_FOUNDATIONAL,
    "n_problems_not_recommended": N_PROBLEMS_NOT_RECOMMENDED,
    "all_hard_gates_passed": bool(AGGREGATION_COMPLETENESS_PASSED and AGGREGATION_SCOPE_CORRECTNESS_PASSED),
    "population_consistency_passed": POPULATION_CONSISTENCY_PASSED,
    "warp_thread_count": WARP_THREAD_COUNT, "max_ram_bytes": MAX_RAM_BYTES, "random_seed": RANDOM_SEED,
}
nb71_summary_path = ARTIFACTS_DIR / "notebook_71_summary.json"
with open(nb71_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_71_summary, f, indent=2)
print(f"Summary written to: {nb71_summary_path}")
print("\n✅ Section 10 complete.")

print(
    f"\n🎯 Notebook 71 (Modeling -- the real BI aggregation layer) complete. "
    f"TOTAL_PLATFORM_NET_VALUE_USD = ${TOTAL_PLATFORM_NET_VALUE_USD:,.2f}/cycle across "
    f"{N_PROBLEMS_RECOMMENDED} recommended value-creation problems. Both hard-gating KPIs passed. "
    f"Next: Notebook 72 (Validation & Deployment)."
)
